In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install opencv-python
!pip install pycocotools

In [3]:
import os
import cv2
import numpy as np
from pycocotools.coco import COCO
from tqdm import tqdm
import json

def create_all_masks(data_dir, split="train2017"):
    """Create mask images for ALL COCO annotation files"""

    # Paths
    img_dir = os.path.join(data_dir, split)
    ann_dir = os.path.join(data_dir, "annotations")
    mask_dir = os.path.join(data_dir, f"{split}_masks")
    os.makedirs(mask_dir, exist_ok=True)

    # All annotation files to process
    ann_files = {
        "instances": f"instances_{split}.json",
        "keypoints": f"person_keypoints_{split}.json",
        "captions": f"captions_{split}.json"  # No masks, just for completeness
    }

    # COCO category names for labeling
    COCO_CLASSES = {1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'airplane',
                   6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light',
                   # ... full 80 classes (add as needed)
                   }

    stats = {"total_images": 0, "total_instances": 0}

    # Process each annotation file type
    for ann_type, ann_file in ann_files.items():
        ann_path = os.path.join(ann_dir, ann_file)
        if not os.path.exists(ann_path):
            print(f"Skipping {ann_file} - not found")
            continue

        print(f"\n🔄 Processing {ann_file}...")
        coco = COCO(ann_path)

        # Get all images
        img_ids = coco.getImgIds()
        stats["total_images"] += len(img_ids)

        # Subdirectory for this annotation type
        type_mask_dir = os.path.join(mask_dir, ann_type)
        os.makedirs(type_mask_dir, exist_ok=True)

        for img_id in tqdm(img_ids, desc=f"{ann_type} masks"):
            img_info = coco.loadImgs(img_id)[0]
            file_name = img_info['file_name']
            h, w = img_info['height'], img_info['width']

            # Get annotations for this image
            ann_ids = coco.getAnnIds(imgIds=img_id)
            anns = coco.loadAnns(ann_ids)
            stats["total_instances"] += len(anns)

            base_name = os.path.splitext(file_name)[0]

            # 1. PER-INSTANCE MASKS (one PNG per object)
            for i, ann in enumerate(anns):
                # Convert polygon/RLE to binary mask
                instance_mask = coco.annToMask(ann)  # [H, W] binary

                # Save individual instance mask
                out_path = os.path.join(type_mask_dir, f"{base_name}_ann{ann['id']}.png")
                cv2.imwrite(out_path, instance_mask * 255)

            # 2. SEMANTIC MASK (all instances merged by class)
            semantic_mask = np.zeros((h, w), dtype=np.uint8)
            for ann in anns:
                mask = coco.annToMask(ann)
                cat_id = ann['category_id']
                semantic_mask[mask > 0] = cat_id  # Overwrite with class ID

            # Save semantic mask
            semantic_path = os.path.join(type_mask_dir, f"{base_name}_semantic.png")
            cv2.imwrite(semantic_path, semantic_mask)

            # 3. CROWD MASK (separate for iscrowd=1)
            crowd_mask = np.zeros((h, w), dtype=np.uint8)
            for ann in anns:
                if ann.get('iscrowd', 0) == 1:
                    mask = coco.annToMask(ann)
                    crowd_mask[mask > 0] = 255

            if np.any(crowd_mask):
                crowd_path = os.path.join(type_mask_dir, f"{base_name}_crowd.png")
                cv2.imwrite(crowd_path, crowd_mask)

    print(f"\n✅ Completed! Stats: {stats}")
    print(f"📁 Masks saved to: {mask_dir}")

# Usage
if __name__ == "__main__":
    data_dir = "/content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017"
    create_all_masks(data_dir, "train2017")
    create_all_masks(data_dir, "val2017")


Skipping instances_train2017.json - not found
Skipping person_keypoints_train2017.json - not found
Skipping captions_train2017.json - not found

✅ Completed! Stats: {'total_images': 0, 'total_instances': 0}
📁 Masks saved to: /content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017\train2017_masks
Skipping instances_val2017.json - not found
Skipping person_keypoints_val2017.json - not found
Skipping captions_val2017.json - not found

✅ Completed! Stats: {'total_images': 0, 'total_instances': 0}
📁 Masks saved to: /content/drive/MyDrive/AI_Vision_Extract_Nov25/data/COCO2017\val2017_masks
